<a href="https://colab.research.google.com/github/asmaatefomran/generative-ai-tasks/blob/main/Task2_ai_agent_dynamic_task_execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.5 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


In [3]:
from groq import Groq
from pydantic import BaseModel
from typing import Optional, List
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import re


In [4]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])


In [9]:
from datetime import datetime
from zoneinfo import ZoneInfo

events = []

def schedule_tool(date, time, event_name, action="create"):
    if action == "create":
        for event in events:
            if event["date"] == date and event["time"] == time:
                return {
                    "success": False,
                    "message": f"Conflict with event: {event['event_name']}"
                }

        event = {
            "date": date,
            "time": time,
            "event_name": event_name
        }

        events.append(event)

        return {
            "success": True,
            "message": "Event created",
            "event": event
        }

    return {"success": False, "message": "Unsupported action"}


def location_tool(location):
    locations = {
        "cairo": ("Egypt", "Africa/Cairo"),
        "london": ("United Kingdom", "Europe/London"),
        "tokyo": ("Japan", "Asia/Tokyo"),
        "dubai": ("United Arab Emirates", "Asia/Dubai")
    }

    location = location.lower()

    if location not in locations:
        return {
            "success": False,
            "message": "Unknown location"
        }

    country, timezone = locations[location]

    return {
        "success": True,
        "country": country,
        "timezone": timezone,
        "current_local_time": datetime.now(
            ZoneInfo(timezone)
        ).strftime("%Y-%m-%d %H:%M:%S")
    }


def analytics_tool(numbers):
    if not numbers:
        return {
            "success": False,
            "message": "Empty list"
        }

    return {
        "success": True,
        "average": sum(numbers) / len(numbers),
        "maximum": max(numbers),
        "minimum": min(numbers),
        "count": len(numbers)
    }


In [10]:
def understand_request(user_input):

    prompt = f"""
You are an AI agent.

Understand the user's request and select one of these tools:

- schedule
- location
- analytics

Return ONLY JSON.

For schedule:
{{
    "intent": "schedule",
    "date": "...",
    "time": "...",
    "event_name": "..."
}}

For location:
{{
    "intent": "location",
    "location": "..."
}}

For analytics:
{{
    "intent": "analytics",
    "numbers": [...]
}}

User request:
{user_input}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return json.loads(response.choices[0].message.content)


In [11]:
def execute_tool(request):

    if request["intent"] == "schedule":
        return schedule_tool(
            request["date"],
            request["time"],
            request["event_name"]
        )

    elif request["intent"] == "location":
        return location_tool(
            request["location"]
        )

    elif request["intent"] == "analytics":
        return analytics_tool(
            request["numbers"]
        )

    return {
        "success": False,
        "message": "Unknown request"
    }


In [12]:
def final_response(user_input, tool_result):

    prompt = f"""
User request:
{user_input}

Tool result:
{json.dumps(tool_result)}

Give the user a clear and concise answer based on the tool result.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


In [15]:
while True:

    user_input = input("\nEnter your request (or type 'exit' to stop): ")

    if user_input.lower() == "exit":
        print("Agent stopped.")
        break

    request = understand_request(user_input)

    print("\nLLM Decision:")
    print(json.dumps(request, indent=2))

    result = execute_tool(request)

    print("\nTool Result:")
    print(json.dumps(result, indent=2))

    answer = final_response(user_input, result)

    print("\nFinal Answer:")
    print(answer)



Enter your request (or type 'exit' to stop): Enter your request (or type 'exit' to stop): Schedule a project meeting on 2026-09-01 at 10:00 called Project Meeting

LLM Decision:
{
  "intent": "schedule",
  "date": "2026-09-01",
  "time": "10:00",
  "event_name": "Project Meeting"
}

Tool Result:
{
  "success": false,
  "message": "Conflict with event: Project Meeting"
}

Final Answer:
I wasn’t able to create the meeting because there’s already an event named **“Project Meeting”** at that date and time.  

You could either:

* Choose a different time or date, or  
* Use a different title for the new meeting.

Let me know how you’d like to proceed!

Enter your request (or type 'exit' to stop): Enter your request (or type 'exit' to stop): What is the current time in Tokyo?

LLM Decision:
{
  "intent": "location",
  "location": "Tokyo"
}

Tool Result:
{
  "success": true,
  "country": "Japan",
  "timezone": "Asia/Tokyo",
  "current_local_time": "2026-08-28 00:32:43"
}

Final Answer:
The c